# 6.24 - FACE hyperparameter heuristic diagnostics

Start small: this notebook only loads the libraries and the tabular datasets used by `benchmark_full.yaml`.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from counterfactuals.datasets.loaders import (
    AdultDataset,
    CompasDataset,
    GermanCreditDataset,
    GiveMeSomeCreditDataset,
    HELOCDataset,
    LendingClubDataset,
    WisconsinBreastCancerDataset,
)
from counterfactuals.utils.config import read_yaml

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)


In [2]:
BENCHMARK_CFG = ROOT / 'configs' / 'benchmarks' / 'full.yaml'
CFG = read_yaml(BENCHMARK_CFG)

DATASET_LOADERS = {
    'adult': AdultDataset,
    'compas': CompasDataset,
    'german_credit': GermanCreditDataset,
    'heloc': HELOCDataset,
    'give_me_some_credit': GiveMeSomeCreditDataset,
    'lending_club': LendingClubDataset,
    'wisconsin_breast_cancer': WisconsinBreastCancerDataset,
}

DATASET_NAMES = [d['name'] for d in CFG['datasets']]
SUBSAMPLE_N = 10000
SUBSAMPLE_SEED = 42
FACE_NORM = 2
TARGET_LARGEST_COMPONENT_RATIO = 0.90
MAX_ISOLATED_FRACTION = 0.05
MIN_AVG_DEGREE = 5.0
MAX_AVG_DEGREE = 100.0
DATASET_NAMES


['wisconsin_breast_cancer',
 'adult',
 'compas',
 'german_credit',
 'heloc',
 'give_me_some_credit',
 'lending_club']

In [3]:
rng = np.random.default_rng(SUBSAMPLE_SEED)
bundles = {}
rows = []

for dataset_name in DATASET_NAMES:
    loader = DATASET_LOADERS[dataset_name](data_dir=str(ROOT / 'data'), seed=42)
    loader.load()
    x_train, y_train = loader.get_train()
    x_test, y_test = loader.get_test()
    n_take = min(SUBSAMPLE_N, len(x_train))
    sample_idx = np.sort(rng.choice(len(x_train), size=n_take, replace=False))
    x_train_sample = x_train[sample_idx]
    y_train_sample = y_train[sample_idx]
    bundles[dataset_name] = {
        'x_train_full': x_train,
        'y_train_full': y_train,
        'x_train': x_train_sample,
        'y_train': y_train_sample,
        'x_test': x_test,
        'y_test': y_test,
        'sample_idx': sample_idx,
    }
    rows.append({
        'dataset': dataset_name,
        'n_train_full': int(len(x_train)),
        'n_train_sample': int(len(x_train_sample)),
        'n_test': int(len(x_test)),
        'n_features': int(x_train.shape[1]),
        'class0_train_fraction': float(np.mean(y_train_sample == 0)),
        'class1_train_fraction': float(np.mean(y_train_sample == 1)),
    })

DATASET_SUMMARY_DF = pd.DataFrame(rows).sort_values('dataset').reset_index(drop=True)
display(DATASET_SUMMARY_DF)


,dataset,n_train_full,n_train_sample,n_test,n_features,class0_train_fraction,class1_train_fraction
0,adult,36178,10000,4522,104,0.745600,0.254400
1,compas,4938,4938,617,14,0.547388,0.452612
2,german_credit,800,800,100,61,0.692500,0.307500
3,give_me_some_credit,120000,10000,15000,10,0.934400,0.065600
4,heloc,8369,8369,1045,23,0.476640,0.523360
5,lending_club,30823,10000,3852,25,0.853900,0.146100
6,wisconsin_breast_cancer,457,457,56,30,0.619256,0.380744


In [4]:
from scipy.sparse.csgraph import connected_components
from sklearn.neighbors import radius_neighbors_graph

EPSILON_GRID = list(np.arange(0.1, 5.1, 0.1)) # Works for L2
# EPSILON_GRID = np.arange(3.5, 25.5, 0.5) # Not working for L1 yet

def sklearn_metric_kwargs(norm: int | float | str) -> dict:
    if norm == 1:
        return {'metric': 'minkowski', 'p': 1}
    if norm == 2:
        return {'metric': 'minkowski', 'p': 2}
    if norm in {'inf', np.inf}:
        return {'metric': 'chebyshev'}
    raise ValueError('Supported norms are 1, 2, and inf.')

def graph_metrics(x: np.ndarray, epsilon: float, norm: int | float | str) -> dict:
    adj = radius_neighbors_graph(
        x,
        radius=float(epsilon),
        mode="connectivity",
        include_self=False,
        **sklearn_metric_kwargs(norm),
    )
    degrees = np.asarray(adj.getnnz(axis=1)).astype(np.int64)
    n_components, labels = connected_components(adj, directed=False, return_labels=True)
    comp_sizes = np.bincount(labels, minlength=n_components) if n_components > 0 else np.array([], dtype=np.int64)
    largest_component = int(comp_sizes.max()) if comp_sizes.size else 0
    isolated_nodes = int(np.sum(degrees == 0)) if len(degrees) else 0
    return {
        "norm": str(norm),
        "epsilon": float(epsilon),
        "n_nodes": int(x.shape[0]),
        "n_edges": int(adj.nnz // 2),
        "n_components": int(n_components),
        "isolated_nodes": isolated_nodes,
        "isolated_fraction": float(isolated_nodes / x.shape[0]) if x.shape[0] else 0.0,
        "largest_component": largest_component,
        "largest_component_ratio": float(largest_component / x.shape[0]) if x.shape[0] else 0.0,
        "avg_degree": float(degrees.mean()) if len(degrees) else 0.0,
        "median_degree": float(np.median(degrees)) if len(degrees) else 0.0,
        "max_degree": int(degrees.max()) if len(degrees) else 0,
    }


In [5]:
graph_rows = []

for dataset_name, bundle in bundles.items():
    x = bundle["x_train"]
    for epsilon in EPSILON_GRID:
        row = {"dataset": dataset_name}
        row.update(graph_metrics(x, epsilon, FACE_NORM))
        graph_rows.append(row)

eps_ok = lambda row: (
    (row['largest_component_ratio'] >= TARGET_LARGEST_COMPONENT_RATIO)
    and (row['isolated_fraction'] <= MAX_ISOLATED_FRACTION)
    and (row['avg_degree'] >= MIN_AVG_DEGREE)
    and (row['avg_degree'] <= MAX_AVG_DEGREE)
)
GRAPH_METRICS_DF = pd.DataFrame(graph_rows).sort_values(["dataset", "epsilon"]).reset_index(drop=True)
GRAPH_METRICS_DF['eps_ok'] = GRAPH_METRICS_DF.apply(eps_ok, axis=1)
GRAPH_METRICS_DF = GRAPH_METRICS_DF[[
    'dataset', 'norm', 'eps_ok', 'epsilon', 'n_nodes', 'n_edges', 'n_components',
    'isolated_nodes', 'isolated_fraction', 'largest_component',
    'largest_component_ratio', 'avg_degree', 'median_degree', 'max_degree'
]]
display(GRAPH_METRICS_DF)


,dataset,norm,eps_ok,epsilon,n_nodes,n_edges,n_components,isolated_nodes,isolated_fraction,largest_component,largest_component_ratio,avg_degree,median_degree,max_degree
0,adult,2,False,0.1,10000,186,9826,9683,0.968300,5,0.000500,0.037200,0.0,4
1,adult,2,False,0.2,10000,577,9539,9252,0.925200,40,0.004000,0.115400,0.0,8
2,adult,2,False,0.3,10000,1302,9169,8747,0.874700,143,0.014300,0.260400,0.0,17
3,adult,2,False,0.4,10000,2386,8732,8193,0.819300,160,0.016000,0.477200,0.0,23
4,adult,2,False,0.5,10000,4203,8110,7523,0.752300,217,0.021700,0.840600,0.0,29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,wisconsin_breast_cancer,2,False,4.6,457,25495,20,19,0.041575,438,0.958425,111.575492,108.0,256
346,wisconsin_breast_cancer,2,False,4.7,457,27110,20,19,0.041575,438,0.958425,118.643326,120.0,267
347,wisconsin_breast_cancer,2,False,4.8,457,28735,20,19,0.041575,438,0.958425,125.754923,129.0,271
348,wisconsin_breast_cancer,2,False,4.9,457,30301,19,18,0.039387,439,0.960613,132.608315,139.0,275


In [7]:
MIN_EPSILON_DF = (
    GRAPH_METRICS_DF[GRAPH_METRICS_DF['eps_ok']]
    .sort_values(['dataset', 'epsilon'])
    .groupby('dataset', as_index=False)
    .first()[['dataset', 'norm', 'epsilon', 'avg_degree', 'isolated_fraction', 'largest_component_ratio', 'n_components']]
    .rename(columns={'epsilon': 'min_valid_epsilon'})
)
display(MIN_EPSILON_DF.round(2))


,dataset,norm,min_valid_epsilon,avg_degree,isolated_fraction,largest_component_ratio,n_components
0,german_credit,2,3.6,26.07,0.05,0.95,41


In [ ]:
E